# Notebook 04 — LangSmith Evaluation
**Goal:** Build an evaluation dataset and measure RAG quality with LangSmith.

By the end of this notebook you will have:
- Created a small evaluation dataset (question + expected answer pairs)
- Run the RAG chain against the eval set
- Scored outputs for correctness and faithfulness using GPT as a judge
- Logged all results to LangSmith for inspection

**Prerequisites:** Run notebooks 01 and 02 first (ingest videos + verify RAG works).

## Step 1 — Environment check

In [ ]:
import sys, os
sys.path.append('..')

from src.utils.config import (
    OPENAI_API_KEY,
    OPENAI_LLM_MODEL,
    OPENAI_EMBEDDING_MODEL,
    PINECONE_API_KEY,
    PINECONE_INDEX_NAME,
    LANGCHAIN_API_KEY,
    LANGCHAIN_PROJECT,
    TOP_K_RESULTS,
)

# Ensure LangSmith tracing is active
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = LANGCHAIN_API_KEY
os.environ['LANGCHAIN_PROJECT'] = LANGCHAIN_PROJECT

print('✅ OpenAI key loaded:', OPENAI_API_KEY[:8] + '...')
print(f'✅ LangSmith project: {LANGCHAIN_PROJECT}')
print(f'✅ Tracing enabled: {os.environ.get("LANGCHAIN_TRACING_V2")}')

## Step 2 — Build the RAG chain
Same chain as notebook 02 — needed as the target for evaluation.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL,
    openai_api_key=OPENAI_API_KEY,
)
vectorstore = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={'k': TOP_K_RESULTS})

llm = ChatOpenAI(
    model=OPENAI_LLM_MODEL,
    openai_api_key=OPENAI_API_KEY,
    temperature=0,
)

RAG_PROMPT = ChatPromptTemplate.from_template(
    """You are a Creative Intelligence Copilot. Answer the question based ONLY on the
following transcript excerpts. If the answer is not in the context, say so.
Always cite which video/source the information comes from.

Context:
{context}

Question: {question}

Answer:"""
)


def format_docs(docs):
    return '\n\n---\n\n'.join(
        f"[Source: {d.metadata.get('title', 'Unknown')}]\n{d.page_content}"
        for d in docs
    )


rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print('✅ RAG chain built.')

## Step 3 — Define the evaluation dataset
Each example has a question and expected answer. Adjust these based on the videos you ingested.

**Tip:** After running notebook 01 with a real video, come back here and update the expected answers to match the actual transcript content.

In [ ]:
eval_examples = [
    {
        'question': 'What skincare products are mentioned in the videos?',
        'expected': 'The answer should list specific skincare products or ingredients mentioned in the ingested transcripts.',
    },
    {
        'question': 'What hook or opening does the creator use?',
        'expected': 'The answer should describe the opening lines or attention-grabbing technique from the video transcript.',
    },
    {
        'question': 'Does the creator make any medical or treatment claims?',
        'expected': 'The answer should identify any claims about curing, treating, or medically addressing skin conditions, or state that none were found.',
    },
    {
        'question': 'What is the main narrative structure of the video?',
        'expected': 'The answer should describe the story arc: problem, journey, solution, or before/after structure from the transcript.',
    },
    {
        'question': 'What call to action does the creator include?',
        'expected': 'The answer should mention any calls to action like subscribing, clicking links, or trying products.',
    },
]

print(f'Evaluation dataset: {len(eval_examples)} examples')
for i, ex in enumerate(eval_examples, 1):
    print(f'  {i}. {ex["question"]}')

## Step 4 — Run the RAG chain on the eval set
Generate answers for each question and store them alongside the expected answers.

In [ ]:
eval_results = []

for i, example in enumerate(eval_examples, 1):
    question = example['question']
    print(f'\n[{i}/{len(eval_examples)}] {question}')
    
    actual_answer = rag_chain.invoke(question)
    
    eval_results.append({
        'question': question,
        'expected': example['expected'],
        'actual': actual_answer,
    })
    
    print(f'  Answer: {actual_answer[:200]}...' if len(actual_answer) > 200 else f'  Answer: {actual_answer}')

print(f'\n✅ Generated answers for {len(eval_results)} questions.')

## Step 5 — Score with GPT-as-judge
Use GPT-4o-mini to evaluate each answer on two dimensions:
- **Correctness:** Does the answer address the question and match the expected answer?
- **Faithfulness:** Is the answer grounded in retrieved context (not hallucinated)?

In [ ]:
import json
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

JUDGE_PROMPT = """You are an evaluation judge. Score the following RAG system answer.

Question: {question}
Expected answer criteria: {expected}
Actual answer: {actual}

Score on two dimensions (1-5 scale):
- correctness: Does the answer address the question and align with the expected criteria?
- faithfulness: Does the answer appear grounded in source material (cites sources, uses specific details) vs. generic/hallucinated?

Respond ONLY with JSON:
{{
  "correctness": <1-5>,
  "faithfulness": <1-5>,
  "reasoning": "<one sentence>"
}}"""

scored_results = []

for i, result in enumerate(eval_results, 1):
    print(f'Scoring {i}/{len(eval_results)}: {result["question"][:60]}...')
    
    response = client.chat.completions.create(
        model=OPENAI_LLM_MODEL,
        messages=[
            {'role': 'system', 'content': 'You are an evaluation judge. Respond only with valid JSON.'},
            {'role': 'user', 'content': JUDGE_PROMPT.format(**result)},
        ],
        temperature=0,
        response_format={'type': 'json_object'},
    )
    
    scores = json.loads(response.choices[0].message.content)
    scored_results.append({**result, **scores})

print('\n✅ All answers scored.')

## Step 6 — Results summary

In [ ]:
print('='*80)
print('EVALUATION RESULTS')
print('='*80)

total_correctness = 0
total_faithfulness = 0

for i, r in enumerate(scored_results, 1):
    print(f'\n--- Q{i}: {r["question"]} ---')
    print(f'  Correctness:  {r["correctness"]}/5')
    print(f'  Faithfulness: {r["faithfulness"]}/5')
    print(f'  Reasoning:    {r["reasoning"]}')
    total_correctness += r['correctness']
    total_faithfulness += r['faithfulness']

n = len(scored_results)
print(f'\n{"="*80}')
print(f'AVERAGES ({n} questions):')
print(f'  Correctness:  {total_correctness/n:.1f}/5')
print(f'  Faithfulness: {total_faithfulness/n:.1f}/5')
print('='*80)

## Step 7 — Log to LangSmith
Create a dataset in LangSmith and log the evaluation run for tracking over time.

After running this cell, visit [smith.langchain.com](https://smith.langchain.com) to view the results.

In [ ]:
from langsmith import Client
from datetime import datetime

ls_client = Client(api_key=LANGCHAIN_API_KEY)

# Create or get the evaluation dataset
dataset_name = f'copilot-eval-{datetime.now().strftime("%Y%m%d")}'

try:
    dataset = ls_client.create_dataset(
        dataset_name=dataset_name,
        description='Evaluation dataset for the Creative Intelligence Copilot RAG chain.',
    )
    print(f'✅ Created LangSmith dataset: {dataset_name}')
except Exception:
    dataset = ls_client.read_dataset(dataset_name=dataset_name)
    print(f'✅ Using existing LangSmith dataset: {dataset_name}')

# Add examples to the dataset
for r in scored_results:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={'question': r['question']},
        outputs={
            'expected': r['expected'],
            'actual': r['actual'],
            'correctness': r['correctness'],
            'faithfulness': r['faithfulness'],
            'reasoning': r['reasoning'],
        },
    )

print(f'\n✅ Logged {len(scored_results)} examples to LangSmith.')
print(f'View at: https://smith.langchain.com/datasets')

## Notes

**Improving scores:**
- Ingest more videos for a richer corpus
- Tune `CHUNK_SIZE` and `TOP_K_RESULTS` in `.env`
- Refine the RAG prompt template

**LangSmith dashboard:** All RAG chain invocations are traced automatically. Check the project dashboard for latency, token usage, and individual trace inspection.

**Re-running evaluations:** Each run creates a timestamped dataset so you can compare quality across iterations.

**Cost estimate:** This notebook makes ~15 GPT-4o-mini calls (5 RAG + 5 judge + a few setup). Total cost: ~$0.01.